# NpuKit — MNIST tiny-ViT (PYNQ-Z2)

Geometry: native **28×28**, patch **7** → **T=16**, patch vec **49→pad56**, **D=16**.

Train on the Docker host (torch):
```bash
python3 host/train_vit_mnist.py
```
Then copy `vit_mnist_weights.npz`, `mnist_sample.npz`, `npukit.bit`, and this notebook to the board.

This notebook checks **ref vs board** match and batch accuracy (not 100% classification).

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("T", vit.VIT_T, "D", vit.VIT_D, "patch_dim", vit.PATCH_DIM_RAW, "->", vit.PATCH_DIM)
print("weights", vit.DEFAULT_WEIGHTS, "exists", vit.DEFAULT_WEIGHTS.exists())
print("sample", vit.DEFAULT_SAMPLE, "exists", vit.DEFAULT_SAMPLE.exists())

T 16 D 16 patch_dim 49 -> 56
weights /home/xilinx/jupyter_notebooks/vit_mnist_weights.npz exists True
sample /home/xilinx/jupyter_notebooks/mnist_sample.npz exists True


## Offline ref (trained weights + real MNIST sample)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=64)
assert rc == 0
print("ref-only return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=64)
=== MNIST tiny-ViT smoke ===
IMG=28 PATCH=7 T=16 D=16 patch_dim=49->pad56 classes=10
scales ACT/W/P=12.81/96.83/132.84
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz

--- image[0] label=4 ---
--- ref ---
ref pred=4 logits_q12[:4]=[-15888, 7544, -40292, -25821]

--- image[1] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-17011, -1817, 17087, 32870]

--- image[2] label=1 ---
--- ref ---
ref pred=1 logits_q12[:4]=[-730, 24384, 9163, -1315]

--- image[3] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-809, -4545, 38869, 19651]

--- image[4] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-7462, -22524, 5060, 26739]

--- image[5] label=5 ---
--- ref ---
ref pred=5 logits_q12[:4]=[-4076, -24348, -7313, 2814]

--- image[6] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[37547, -17788, -1153, -18148]

--- image[7] label=9 ---
--- ref ---
ref pred=9 logits_q12[:4]=[-13639, 10223, -5662, 

## Board: ref vs FPGA + batch accuracy

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=64)
assert rc == 0
print("board return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=64)
=== MNIST tiny-ViT smoke ===
IMG=28 PATCH=7 T=16 D=16 patch_dim=49->pad56 classes=10
scales ACT/W/P=12.81/96.83/132.84
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz


Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=4 ---
--- ref ---
ref pred=4 logits_q12[:4]=[-15888, 7544, -40292, -25821]
--- FPGA ---
hw  pred=4 logits_q12[:4]=[-15651, 7769, -39955, -25986]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=373  tol=1024
logits: PASS  max|err|=380  tol=1024

--- image[1] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-17011, -1817, 17087, 32870]
--- FPGA ---
hw  pred=3 logits_q12[:4]=[-16833, -1926, 17025, 32850]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=373  tol=1024
logits: PASS  max|err|=330  tol=1024

--- image[2] label=1 ---
--- ref ---
ref pred=1 logits_q12[:4]=[-730, 24384, 9163, -1315]
--- FPGA ---
hw  pred=1 logits_q12[:4]=[-730, 24384, 9163, -1315]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=456  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[3] label=2 ---
--- ref ---
ref pred=2 logits_q